In [43]:
import pandas as pd
import numpy as np
from pathlib import Path




PROJECT_DIR = Path(r"C:\Projects\data-science")

modeling_table = pd.read_csv(
    PROJECT_DIR
    / "data"
    / "processed"
    / "steam_modeling_data_large.csv"
)

print(modeling_table.shape)

genre_columns = [
    col for col in modeling_table.columns
    if col.startswith("genre_")
]

feature_cols = [
    "is_free",
    "initial_price_usd",
    "language_count",
    "years_since_2015",
    "mat_supports_windows",
    "mat_supports_mac",
    "mat_supports_linux",
    "asinh_month1_players",
] + genre_columns

target = "asinh_player_change"

development_df = modeling_table[
    modeling_table["release_year"] <= 2021
].copy()

test_df = modeling_table[
    modeling_table["release_year"].between(2022, 2023)
].copy()

partial_2024_df = modeling_table[
    modeling_table["release_year"] == 2024
].copy()

print("Development:", len(development_df))
print("Final test:", len(test_df))
print("Partial 2024:", len(partial_2024_df))

fold_definitions = [
    {
        "name": "Fold 1",
        "train_end": 2016,
        "val_year": 2018,
    },
    {
        "name": "Fold 2",
        "train_end": 2017,
        "val_year": 2019,
    },
    {
        "name": "Fold 3",
        "train_end": 2018,
        "val_year": 2020,
    },
    {
        "name": "Fold 4",
        "train_end": 2019,
        "val_year": 2021,
    },
]

for fold in fold_definitions:

    train = development_df[
        development_df["release_year"] <= fold["train_end"]
    ]

    val = development_df[
        development_df["release_year"] == fold["val_year"]
    ]

    print(
        fold["name"],
        "| Train:", len(train),
        "| Validation:", len(val)
    )

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================================
# Evaluation function
# ============================================================

def evaluate_model_over_time(
    model_name,
    model,
    data,
    feature_cols,
    target,
    fold_definitions
):
    results = []

    for fold in fold_definitions:

        train_df = data[
            data["release_year"] <= fold["train_end"]
        ].copy()

        val_df = data[
            data["release_year"] == fold["val_year"]
        ].copy()

        X_train = train_df[feature_cols]
        y_train = train_df[target]

        X_val = val_df[feature_cols]
        y_val = val_df[target]

        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", model)
        ])

        pipeline.fit(X_train, y_train)

        predictions = pipeline.predict(X_val)

        mae = mean_absolute_error(y_val, predictions)

        rmse = np.sqrt(
            mean_squared_error(y_val, predictions)
        )

        r2 = r2_score(y_val, predictions)

        results.append({
            "model": model_name,
            "fold": fold["name"],
            "train_end": fold["train_end"],
            "validation_year": fold["val_year"],
            "train_size": len(train_df),
            "validation_size": len(val_df),
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

    return pd.DataFrame(results)


# ============================================================
# Dummy baseline
# ============================================================

dummy_results = evaluate_model_over_time(
    model_name="Dummy Median",
    model=DummyRegressor(strategy="median"),
    data=development_df,
    feature_cols=feature_cols,
    target=target,
    fold_definitions=fold_definitions
)


# ============================================================
# Linear Regression
# ============================================================

linear_results = evaluate_model_over_time(
    model_name="Linear Regression",
    model=LinearRegression(),
    data=development_df,
    feature_cols=feature_cols,
    target=target,
    fold_definitions=fold_definitions
)


# ============================================================
# Random Forest
# ============================================================

rf_results = evaluate_model_over_time(
    model_name="Random Forest",
    model=RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),
    data=development_df,
    feature_cols=feature_cols,
    target=target,
    fold_definitions=fold_definitions
)


# Confirm they were created
print("Dummy results:", dummy_results.shape)
print("Linear results:", linear_results.shape)
print("Random Forest results:", rf_results.shape)

baseline_results = pd.concat(
    [
        dummy_results,
        linear_results,
        rf_results
    ],
    ignore_index=True
)

baseline_results

model_comparison = (
    baseline_results
    .groupby("model")
    .agg(
        mean_MAE=("MAE", "mean"),
        std_MAE=("MAE", "std"),
        mean_RMSE=("RMSE", "mean"),
        std_RMSE=("RMSE", "std"),
        mean_R2=("R2", "mean"),
        std_R2=("R2", "std"),
    )
    .sort_values("mean_MAE")
)

model_comparison

baseline_results.pivot(
    index="validation_year",
    columns="model",
    values="MAE"
)

RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

baseline_results.to_csv(
    RESULTS_DIR / "baseline_cv_results.csv",
    index=False
)

model_comparison.to_csv(
    RESULTS_DIR / "model_comparison.csv"
)

per_year_mae = baseline_results.pivot(
    index="validation_year",
    columns="model",
    values="MAE"
)

per_year_mae

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

def evaluate_ridge_over_time(
    alpha,
    data,
    feature_cols,
    target,
    fold_definitions
):
    results = []

    for fold in fold_definitions:

        train_df = data[
            data["release_year"] <= fold["train_end"]
        ].copy()

        val_df = data[
            data["release_year"] == fold["val_year"]
        ].copy()

        X_train = train_df[feature_cols]
        y_train = train_df[target]

        X_val = val_df[feature_cols]
        y_val = val_df[target]

        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=alpha))
        ])

        pipeline.fit(X_train, y_train)

        predictions = pipeline.predict(X_val)

        results.append({
            "model": f"Ridge alpha={alpha}",
            "fold": fold["name"],
            "validation_year": fold["val_year"],
            "MAE": mean_absolute_error(y_val, predictions),
            "RMSE": np.sqrt(
                mean_squared_error(y_val, predictions)
            ),
            "R2": r2_score(y_val, predictions)
        })

    return pd.DataFrame(results)


# Run Ridge with alpha = 1
ridge_results = evaluate_ridge_over_time(
    alpha=1.0,
    data=development_df,
    feature_cols=feature_cols,
    target=target,
    fold_definitions=fold_definitions
)

ridge_results

ridge_results

ridge_results[
    ["MAE", "RMSE", "R2"]
].agg(["mean", "std"])


all_results = pd.concat(
    [
        baseline_results,
        ridge_results
    ],
    ignore_index=True
)

all_results.pivot(
    index="validation_year",
    columns="model",
    values="MAE"
)

ridge_results[
    ["MAE", "RMSE", "R2"]
].agg(["mean", "std"])


from sklearn.ensemble import GradientBoostingRegressor

gb_results = evaluate_model_over_time(
    model_name="Gradient Boosting",
    model=GradientBoostingRegressor(
        random_state=42
    ),
    data=development_df,
    feature_cols=feature_cols,
    target=target,
    fold_definitions=fold_definitions
)

gb_results


all_results = pd.concat(
    [
        dummy_results,
        linear_results,
        ridge_results,
        rf_results,
        gb_results
    ],
    ignore_index=True
)

full_comparison = (
    all_results
    .groupby("model")
    .agg(
        mean_MAE=("MAE", "mean"),
        std_MAE=("MAE", "std"),
        mean_RMSE=("RMSE", "mean"),
        std_RMSE=("RMSE", "std"),
        mean_R2=("R2", "mean"),
        std_R2=("R2", "std")
    )
    .sort_values("mean_MAE")
)

full_comparison

all_results.pivot(
    index="validation_year",
    columns="model",
    values="MAE"
)

full_comparison = (
    all_results
    .groupby("model")
    .agg(
        mean_MAE=("MAE", "mean"),
        std_MAE=("MAE", "std"),
        mean_RMSE=("RMSE", "mean"),
        std_RMSE=("RMSE", "std"),
        mean_R2=("R2", "mean"),
        std_R2=("R2", "std")
    )
    .sort_values("mean_MAE")
)

full_comparison

full_comparison.to_csv(
    RESULTS_DIR / "model_comparison.csv"
)


gb_parameter_grid = [
    {
        "n_estimators": 100,
        "learning_rate": 0.05,
        "max_depth": 2
    },
    {
        "n_estimators": 100,
        "learning_rate": 0.05,
        "max_depth": 3
    },
    {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "max_depth": 2
    },
    {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "max_depth": 3
    },
    {
        "n_estimators": 200,
        "learning_rate": 0.05,
        "max_depth": 2
    },
    {
        "n_estimators": 200,
        "learning_rate": 0.05,
        "max_depth": 3
    }
]

gb_tuning_results = []

for params in gb_parameter_grid:

    model = GradientBoostingRegressor(
        n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        max_depth=params["max_depth"],
        random_state=42
    )

    results = evaluate_model_over_time(
        model_name=(
            f"GB_n{params['n_estimators']}"
            f"_lr{params['learning_rate']}"
            f"_d{params['max_depth']}"
        ),
        model=model,
        data=development_df,
        feature_cols=feature_cols,
        target=target,
        fold_definitions=fold_definitions
    )

    results["n_estimators"] = params["n_estimators"]
    results["learning_rate"] = params["learning_rate"]
    results["max_depth"] = params["max_depth"]

    gb_tuning_results.append(results)

gb_tuning_results = pd.concat(
    gb_tuning_results,
    ignore_index=True
)

gb_tuning_summary = (
    gb_tuning_results
    .groupby(
        [
            "n_estimators",
            "learning_rate",
            "max_depth"
        ]
    )
    .agg(
        mean_MAE=("MAE", "mean"),
        std_MAE=("MAE", "std"),
        mean_RMSE=("RMSE", "mean"),
        mean_R2=("R2", "mean")
    )
    .reset_index()
    .sort_values("mean_MAE")
)

gb_tuning_summary
test_df

print(test_df["asinh_player_change"].isna().sum())

(4088, 28)
Development: 3950
Final test: 109
Partial 2024: 29
Fold 1 | Train: 1755 | Validation: 637
Fold 2 | Train: 2862 | Validation: 241
Fold 3 | Train: 3499 | Validation: 123
Fold 4 | Train: 3740 | Validation: 87
Dummy results: (4, 9)
Linear results: (4, 9)
Random Forest results: (4, 9)
0
